In [5]:
# ==========================================================
# Smart Student Performance Prediction System
# Task 1: Regression (Final Marks) | Task 2: Classification (Pass/Fail)
# Task 3: Model Comparison + Interactive Student Input & Dataset Update
# ==========================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, confusion_matrix,
                             classification_report)

print("ML environment ready")

# ==========================================================
# 1. Load Dataset & Basic Preprocessing (20 Records)
# ==========================================================
data = {
    "Study Hours":    (2, 3, 5, 1, 6, 4, 2, 7, 3, 8, 5, 1,
                       4, 2.5, 6.5, 3.5, 1.5, 7.5, 5.5, 4.5),
    "Attendance (%)": (60, 65, 75, 40, 85, 70, 50, 90, 55, 95, 80, 35,
                       72, 58, 88, 68, 45, 92, 78, 74),
    "Previous Marks": (45, 50, 60, 30, 70, 55, 35, 80, 40, 85, 65, 25,
                       58, 42, 74, 52, 32, 82, 62, 57),
    "Final Marks":    (48, 52, 65, 35, 78, 60, 40, 88, 45, 92, 72, 30,
                       63, 47, 81, 57, 38, 90, 68, 62),
    "Result":         ("Fail", "Pass", "Pass", "Fail", "Pass", "Pass",
                       "Fail", "Pass", "Fail", "Pass", "Pass", "Fail",
                       "Pass", "Fail", "Pass", "Pass", "Fail", "Pass",
                       "Pass", "Pass"),
}

df = pd.DataFrame(data)

print("\n--- Step 1: Initial Dataset Inspection ---")
print(df.head())
print(f"\nInitial Dataset Shape: {df.shape} ({len(df)} students)")
print("Missing Values:\n", df.isnull().sum())

# Common input features for both tasks
feature_cols = ["Study Hours", "Attendance (%)", "Previous Marks"]
X = df[feature_cols]

# ==========================================================
# TASK 1 - Regression Model: Predict Student Marks
# ==========================================================
print("\n==========================================")
print("TASK 1: REGRESSION MODEL (LINEAR REGRESSION)")
print("==========================================")

y_reg = df["Final Marks"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.25, random_state=42
)

reg_model = LinearRegression()
reg_model.fit(X_train_r, y_train_r)

y_pred_r = reg_model.predict(X_test_r)
mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

print(f"MAE : {mae:.2f} marks | MSE : {mse:.2f} | R² : {r2:.4f}")

# ==========================================================
# TASK 2 - Classification Model: Predict Pass or Fail
# ==========================================================
print("\n==========================================")
print("TASK 2: CLASSIFICATION MODEL (LOGISTIC REGRESSION)")
print("==========================================")

y_clf = df["Result"].map({"Fail": 0, "Pass": 1})

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.25, random_state=42, stratify=y_clf
)

clf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_model.fit(X_train_c, y_train_c)

y_pred_c = clf_model.predict(X_test_c)
acc = accuracy_score(y_test_c, y_pred_c)

print(f"Accuracy: {acc * 100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test_c, y_pred_c, labels=(0, 1)))
print("\nClassification Report:\n", classification_report(
    y_test_c, y_pred_c, labels=(0, 1), target_names=("Fail", "Pass"), zero_division=0
))

# ==========================================================
# TASK 3 - Compare Both Models
# ==========================================================
print("==========================================")
print("TASK 3: MODEL COMPARISON SUMMARY")
print("==========================================")

comparison = pd.DataFrame({
    "Feature": ("Algorithm Used", "Target Variable", "Prediction Type", "Key Metric Score", "Real-World Use"),
    "Task 1: Regression": (
        "Linear Regression",
        "Final Marks (Numerical)",
        "Continuous Score",
        f"MAE: {mae:.2f} | R²: {r2:.4f}",
        "Forecasting exact exam grades"
    ),
    "Task 2: Classification": (
        "Logistic Regression",
        "Result (Pass=1, Fail=0)",
        "Discrete Category",
        f"Accuracy: {acc * 100:.2f}%",
        "Early-warning alerts for at-risk students"
    )
})
print(comparison.to_string(index=False))

# ==========================================================
# TASK 4 - Interactive Student Input, Prediction & Dataset Append
# ==========================================================
print("\n==========================================")
print("INTERACTIVE NEW STUDENT PREDICTION SYSTEM")
print("==========================================")

while True:
    try:
        num_students = int(input("How many new students do you want to enter? "))
        if num_students >= 0:
            break
        print("Please enter 0 or a positive whole number.")
    except ValueError:
        print("Invalid input. Please enter a whole number (e.g., 2).")

if num_students > 0:
    new_records = list()

    for i in range(1, num_students + 1):
        print(f"\n--- Enter Details for Student {i} ---")
        while True:
            try:
                hours = float(input("  Study Hours (e.g., 5.5): "))
                attendance = float(input("  Attendance % (0-100): "))
                prev_marks = float(input("  Previous Marks (0-100): "))
                break
            except ValueError:
                print("  Invalid number entered. Please try again with numeric values.")

        new_records.append({
            "Study Hours": hours,
            "Attendance (%)": attendance,
            "Previous Marks": prev_marks
        })

    # Convert entered records into a DataFrame for model processing
    new_students_df = pd.DataFrame(new_records)

    # Apply Regression and Classification predictions
    pred_marks = reg_model.predict(new_students_df[feature_cols])
    pred_classes = clf_model.predict(new_students_df[feature_cols])
    pred_probs = clf_model.predict_proba(new_students_df[feature_cols])[:, 1]

    # Attach predictions cleanly to the new students table
    new_students_df["Final Marks"] = np.round(pred_marks, 2)
    new_students_df["Result"] = np.where(pred_classes == 1, "Pass", "Fail")
    new_students_df["Pass Probability (%)"] = np.round(pred_probs * 100, 1)

    print("\n--- Prediction Results for Newly Entered Students ---")
    print(new_students_df.to_string(index=False))

    # Append new student records (matching original 5 columns) to existing dataset
    records_to_append = new_students_df[["Study Hours", "Attendance (%)", "Previous Marks", "Final Marks", "Result"]]
    df = pd.concat((df, records_to_append), ignore_index=True)

    print(f"\nSuccess! {num_students} new record(s) added to the existing dataset.")
    print(f"Updated Dataset Shape: {df.shape} ({len(df)} total students)")
    print("\n--- Updated Dataset (Last 5 Rows) ---")
    print(df.tail())
else:
    print("\nNo new students entered. Dataset remains unchanged at 20 records.")


ML environment ready

--- Step 1: Initial Dataset Inspection ---
   Study Hours  Attendance (%)  Previous Marks  Final Marks Result
0          2.0              60              45           48   Fail
1          3.0              65              50           52   Pass
2          5.0              75              60           65   Pass
3          1.0              40              30           35   Fail
4          6.0              85              70           78   Pass

Initial Dataset Shape: (20, 5) (20 students)
Missing Values:
 Study Hours       0
Attendance (%)    0
Previous Marks    0
Final Marks       0
Result            0
dtype: int64

TASK 1: REGRESSION MODEL (LINEAR REGRESSION)
MAE : 1.34 marks | MSE : 3.48 | R² : 0.9869

TASK 2: CLASSIFICATION MODEL (LOGISTIC REGRESSION)
Accuracy: 100.00%

Confusion Matrix:
 [[2 0]
 [0 3]]

Classification Report:
               precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         2
        Pass       1.00     

How many new students do you want to enter?  2



--- Enter Details for Student 1 ---


  Study Hours (e.g., 5.5):  5.7
  Attendance % (0-100):  89
  Previous Marks (0-100):  90



--- Enter Details for Student 2 ---


  Study Hours (e.g., 5.5):  5.9
  Attendance % (0-100):  90
  Previous Marks (0-100):  79



--- Prediction Results for Newly Entered Students ---
 Study Hours  Attendance (%)  Previous Marks  Final Marks Result  Pass Probability (%)
         5.7            89.0            90.0        99.10   Pass                  99.3
         5.9            90.0            79.0        86.23   Pass                  99.0

Success! 2 new record(s) added to the existing dataset.
Updated Dataset Shape: (22, 5) (22 total students)

--- Updated Dataset (Last 5 Rows) ---
    Study Hours  Attendance (%)  Previous Marks  Final Marks Result
17          7.5            92.0            82.0        90.00   Pass
18          5.5            78.0            62.0        68.00   Pass
19          4.5            74.0            57.0        62.00   Pass
20          5.7            89.0            90.0        99.10   Pass
21          5.9            90.0            79.0        86.23   Pass
